# Deprecated: сравнение KV-cache
Самостоятельный исследовательский ноутбук из исходного архива.
Использует внешние реализации и отдельное окружение; не входит в основной pipeline.


# KV-кэш: Qwen3-8B и пять вариантов декодирования

Проверяем дополнительную память **постоянного KV-кэша отдельного drafter**.
Главный эксперимент: batch=1, bf16, один и тот же префикс ровно из N токенов.
Кэш действительно создаётся прямыми проходами готовых моделей; формула не подменяет измерение.
Дополнительный эксперимент запускает готовые циклы генерации и записывает изменение кэшей.

| Вариант | Чекпоинт / реализация |
|---|---|
| AR | `Qwen/Qwen3-8B`, Transformers |
| Orthrus | `chiennv/Orthrus-Qwen3-8B`, код опубликованного checkpoint |
| DFlash | `deepseek-ai/dflash_qwen3_8b_block7`, DeepSpec |
| DSpark | `deepseek-ai/dspark_qwen3_8b_block7`, DeepSpec |
| EAGLE-3 | `deepseek-ai/eagle3_qwen3_8b_ttt7`, DeepSpec |
| AR + AR-drafter | target Qwen3-8B + Qwen3-0.6B, Transformers assisted generation |

DFlash и EAGLE-3 здесь — выпущенные DeepSpec варианты, обученные для той же Qwen3-8B.
EAGLE-3 использует линейный proposal из реализации DeepSpec, а не tree verifier SafeAILab.
Версии кода и весов закреплены. Нового обучения нет.

**Запуск:** ячейки 1–7 по порядку. Они создают отдельное окружение и запускают
методы последовательно в отдельных процессах на GPU 0. Текущий kernel и его пакеты
не переустанавливаются. При первом запуске будут скачаны веса 8B и draft-моделей.
8B-замеры на GPU не выполнялись при подготовке notebook; ниже есть отдельно помеченная
расчётная таблица. Код проверен на настоящих upstream-классах с маленькими моделями на CPU.

## 1. Настройки
`CONTEXTS` — точные длины общего префикса, включая все его токены. Синтетический повтор текста
нужен для контроля длины: это эксперимент по памяти, не оценка качества ответа.
Для собственного **8B** Orthrus-checkpoint укажи локальный путь в `MODELS['orthrus']['id']`
и `revision=None`. Checkpoint 0.6B из предыдущего бакета имеет другую архитектуру.

In [ ]:
import json
import hashlib
import os
import subprocess
import sys
from pathlib import Path

GPU = "1"
CONTEXTS = [512, 2048, 8192]
DTYPE = "bfloat16"  # Можно выбрать float32; это удвоит KV bytes и размер весов в GPU.
METHODS = ["ar", "orthrus", "dflash", "dspark", "eagle3", "ar_draft"]
RUN_GENERATION = True  # Включи для дополнительного эксперимента с полными decoder loops.
MAX_NEW_TOKENS = 64

ROOT = (Path.cwd() / "qwen3_8b_kv_benchmark").resolve()
ROOT.mkdir(parents=True, exist_ok=True)
ENV_DIR = ROOT / "env"
PYTHON = ENV_DIR / "bin" / "python"
DEEPSPEC_DIR = ROOT / "vendor" / "DeepSpec"
DEEPSPEC_REVISION = "005e03b81cec38b7da6399833d609ee89a2587f2"
# CUDA 12.8 wheel; при необходимости укажи индекс PyTorch для своего драйвера.
TORCH_INDEX = "https://download.pytorch.org/whl/cu128"
CREATE_ENVIRONMENT = True
print("Рабочая папка:", ROOT)
MODELS = {'target': {'id': 'Qwen/Qwen3-8B', 'revision': 'b968826d9c46dd6066d109eabc6255188de91218'}, 'orthrus': {'id': 'chiennv/Orthrus-Qwen3-8B', 'revision': '977a617772e91c966a8cd9b551f4151f9824b6fa'}, 'dflash': {'id': 'deepseek-ai/dflash_qwen3_8b_block7', 'revision': '9e44dbbb6cb68b0c943abf9c5fc3c17c00897cdf'}, 'dspark': {'id': 'deepseek-ai/dspark_qwen3_8b_block7', 'revision': '03326e5043815da1f81b109078b2889737c26017'}, 'eagle3': {'id': 'deepseek-ai/eagle3_qwen3_8b_ttt7', 'revision': 'f6485ba8d21e11942958617dbe7e71b467f38f38'}, 'ar_draft': {'id': 'Qwen/Qwen3-0.6B', 'revision': 'c1899de289a04d12100db370d81485cdf75e47ca'}}

## 2. Отдельное окружение и исходный DeepSpec
Если уже есть отдельное окружение с Torch 2.9.1 и Transformers 5.10.2,
задай его `PYTHON` и поставь `CREATE_ENVIRONMENT=False`. `torchvision`/`torchaudio`
не устанавливаются: они не нужны для этих текстовых моделей.

In [ ]:
if CREATE_ENVIRONMENT:
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "venv", str(ENV_DIR)], check=True)
    subprocess.run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([
        str(PYTHON), "-m", "pip", "install", "torch==2.9.1",
        "--index-url", TORCH_INDEX,
    ], check=True)
    subprocess.run([
        str(PYTHON), "-m", "pip", "install", "transformers==5.10.2",
        "accelerate==1.15.0", "safetensors==0.8.0", "numpy>=2.0,<3",
        "PyYAML==6.0.3", "prettytable==3.18.0", "typing_extensions>=4.15",
    ], check=True)

DEEPSPEC_DIR.parent.mkdir(parents=True, exist_ok=True)
if not DEEPSPEC_DIR.exists():
    subprocess.run([
        "git", "clone", "https://github.com/deepseek-ai/DeepSpec.git", str(DEEPSPEC_DIR)
    ], check=True)
    subprocess.run([
        "git", "-C", str(DEEPSPEC_DIR), "checkout", "--detach", DEEPSPEC_REVISION
    ], check=True)
actual_revision = subprocess.check_output([
    "git", "-C", str(DEEPSPEC_DIR), "rev-parse", "HEAD"
], text=True).strip()
assert actual_revision == DEEPSPEC_REVISION, (actual_revision, DEEPSPEC_REVISION)

worker_env = os.environ.copy()
worker_env["CUDA_VISIBLE_DEVICES"] = GPU
worker_env["PYTHONPATH"] = str(DEEPSPEC_DIR)
worker_env["TOKENIZERS_PARALLELISM"] = "false"
worker_env["PYTHONUNBUFFERED"] = "1"
subprocess.run([
    str(PYTHON), "-c",
    "import torch, transformers; "
    "from deepspec.eval.dspark.evaluator import Qwen3DSparkEvaluator; "
    "print('Torch:', torch.__version__, 'Transformers:', transformers.__version__); "
    "assert torch.cuda.is_available(), 'CUDA не доступна'; "
    "print('GPU:', torch.cuda.get_device_name(0))"
], env=worker_env, check=True)

## 3. Измеритель (служебная ячейка)
Здесь только загрузка, вызовы готовых моделей/декодеров и измерение тензоров.
Реализации attention и speculative decoding не переписываются.

Основные поля:
- `target_tokens`, `draft_tokens`: число позиций в истории соответствующего кэша,
  **без умножения на число слоёв**. Для всех полных слоёв длины проверяются на равенство N.
- `*_layer_slots`: позиции, суммированные по слоям (одна пара K/V на слот).
- `*_bytes`: полезные данные K и V по всем слоям (`numel * element_size`).
- `*_storage_bytes`: размер уникальных backing storages, включая удерживаемый хвост view.
- `total_storage_bytes`: сумма с устранением повторов/алиасов хранилищ.

`sum_cache_positions` считает позиции отдельно хранимых кэшей; это не количество уникальных
токенов текста. Веса моделей, hidden states, временные attention tensors, allocator reserve
не входят в KV bytes. После `crop` backing storage может остаться больше полезных данных.

In [ ]:
WORKER_SOURCE = r'''
"""Measure real KV tensors using pinned upstream implementations; no decoder reimplementation."""
from __future__ import annotations

import argparse
import contextlib
import gc
import importlib
import json
import sys
import time
import weakref
from pathlib import Path
from types import SimpleNamespace

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache


def cache_tensors(cache):
    if cache is None:
        return []
    if hasattr(cache, "layers"):
        return [(x.keys, x.values) for x in cache.layers
                if getattr(x, "keys", None) is not None and getattr(x, "values", None) is not None]
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    if isinstance(cache, (tuple, list)):
        return [(x[0], x[1]) for x in cache if len(x) >= 2]
    raise TypeError(f"Unsupported cache class: {type(cache)}")


def measure_cache(cache):
    pairs = cache_tensors(cache)
    if not pairs:
        return dict(tokens=0, layers=0, layer_slots=0, bytes=0, storage_bytes=0,
                    dtype=None, lengths=[], storages={})
    lengths = [int(k.shape[-2]) for k, _ in pairs]
    batches = [int(k.shape[0]) for k, _ in pairs]
    storages = {}
    for pair in pairs:
        for tensor in pair:
            if not isinstance(tensor, torch.Tensor):
                raise TypeError(type(tensor))
            storage = tensor.untyped_storage()
            key = f"{tensor.device}:{storage.data_ptr()}"
            storages[key] = int(storage.nbytes())
    return dict(
        tokens=max(lengths), layers=len(pairs),
        layer_slots=sum(n * b for n, b in zip(lengths, batches)),
        bytes=sum(t.numel() * t.element_size() for pair in pairs for t in pair),
        storage_bytes=sum(storages.values()),
        dtype=str(pairs[0][0].dtype), lengths=lengths, storages=storages,
    )


def combined_cache(target_cache, draft_cache):
    target, draft = measure_cache(target_cache), measure_cache(draft_cache)
    row = {}
    for role, values in [("target", target), ("draft", draft)]:
        for key in ("tokens", "layers", "layer_slots", "bytes", "storage_bytes", "dtype"):
            row[f"{role}_{key}"] = values[key]
    row["sum_cache_positions"] = target["tokens"] + draft["tokens"]
    row["total_bytes"] = target["bytes"] + draft["bytes"]
    row["total_storage_bytes"] = sum({**target["storages"], **draft["storages"]}.values())
    row["target_layer_lengths"] = target["lengths"]
    row["draft_layer_lengths"] = draft["lengths"]
    return row


def make_evaluator(method, target, draft, max_new_tokens=64):
    # Reuse the public decoder methods without launching distributed dataset evaluation.
    if method in ("dflash", "dspark"):
        from deepspec.eval.dspark.evaluator import Qwen3DSparkEvaluator
        cls = Qwen3DSparkEvaluator
    elif method == "eagle3":
        from deepspec.eval.eagle3.evaluator import Qwen3Eagle3Evaluator
        cls = Qwen3Eagle3Evaluator
    else:
        raise ValueError(method)
    obj = cls.__new__(cls)
    obj.target_model, obj.draft_model = target, draft
    obj.device = next(target.parameters()).device
    obj.args = SimpleNamespace(temperature=0.0, confidence_threshold=0.0,
                               max_new_tokens=max_new_tokens)
    obj.confidence_head_recorder = None
    return obj


@torch.inference_mode()
def measure_prefix(method, target, draft, input_ids, probe_block=True):
    """Exactly N committed prefix positions; never count rejected proposals as prefix."""
    device, n = input_ids.device, int(input_ids.shape[1])
    target_cache = DynamicCache(config=target.config)
    deep = method in ("dflash", "dspark", "eagle3")
    output = target(input_ids=input_ids, past_key_values=target_cache, use_cache=True,
                    logits_to_keep=1, output_hidden_states=deep)
    next_token = output.logits[:, -1:].argmax(-1)
    draft_cache = None
    temporary_positions = 0
    if deep:
        evaluator = make_evaluator(method, target, draft)
        block = int(evaluator.max_proposal_tokens)
        positions = torch.arange(n + block + 2, device=device)[None]
        output_ids = torch.zeros((1, n + block + 2), dtype=torch.long, device=device)
        output_ids[:, :n] = input_ids
        output_ids[:, n:n + 1] = next_token
        context = evaluator._init_context(initial_output=output, output_ids=output_ids,
                    position_ids=positions, num_input_tokens=n)
        if method == "eagle3":
            draft_cache = context.draft_cache
            if probe_block:
                proposal = evaluator._propose(context=context, output_ids=output_ids,
                    position_ids=positions, start=n, stop_token_ids=None)
                temporary_positions = int(draft_cache.get_seq_length()) - n
                draft_cache.crop(n)
                del proposal
        else:
            # The released DFlash/DSpark first proposal initializes its own history cache.
            # forward_dspark_draft_block crops the mask block back to the accepted prefix.
            proposal = evaluator._propose(context=context, output_ids=output_ids,
                    position_ids=positions, start=n, stop_token_ids=None)
            draft_cache = context.past_key_values_draft
            temporary_positions = block
            del proposal
    elif method == "ar_draft":
        draft_cache = DynamicCache(config=draft.config)
        draft_output = draft(input_ids=input_ids, past_key_values=draft_cache,
                             use_cache=True, logits_to_keep=1)
        del draft_output
    elif method == "orthrus" and probe_block:
        block_size = int(target.config.block_size)
        block_ids = torch.full((1, block_size), int(target.config.mask_token_id),
                               dtype=torch.long, device=device)
        block_ids[:, :1] = next_token
        diffusion_output = target(input_ids=block_ids,
                    position_ids=torch.arange(n, n + block_size, device=device)[None],
                    past_key_values=target_cache, use_cache=False,
                    is_diffusion_pass=True, ar_seq_len=n)
        del diffusion_output
        temporary_positions = block_size  # Block K/V are temporary, not another history cache.
    row = combined_cache(target_cache, draft_cache)
    if row["target_layer_lengths"] != [n] * target.config.num_hidden_layers:
        raise AssertionError(f"Target prefix is not exactly N={n}: {row}")
    if draft is not None and row["draft_layer_lengths"] != [n] * draft.config.num_hidden_layers:
        raise AssertionError(f"Draft prefix is not exactly N={n}: {row}")
    row.update(mode="fixed_prefix", method=method, context_tokens=n, status="ok",
               temporary_draft_block_positions=temporary_positions)
    return row


class CacheRecorder:
    """Sample existing live caches; keep weak refs, not hidden GPU copies."""
    def __init__(self):
        self.refs = {}
        self.trace = []

    def register(self, role, cache):
        if cache is not None:
            if not hasattr(cache, "layers") and not hasattr(cache, "key_cache"):
                raise TypeError(f"Generation cache must be DynamicCache, got {type(cache)}")
            self.refs[role] = weakref.ref(cache)

    def sample(self, event):
        caches = {role: ref() for role, ref in self.refs.items()}
        row = combined_cache(caches.get("target"), caches.get("draft"))
        row.update(event=event, sample=len(self.trace))
        self.trace.append(row)

    @contextlib.contextmanager
    def instrument(self, target, draft):
        handles = []
        def make_pre(role):
            def pre(module, args, kwargs):
                cache = kwargs.get("past_key_values", kwargs.get("past_key_value"))
                self.register(role, cache)
            return pre

        def make_post(role):
            def post(module, args, kwargs, output):
                cache = kwargs.get("past_key_values", kwargs.get("past_key_value"))
                if cache is None:
                    cache = getattr(output, "past_key_values", None)
                self.register(role, cache)
                self.sample(role + ":after_forward")
            return post

        modules = [("target", target)]
        if draft is not None:
            modules.append(("draft", draft))
            # DFlash/DSpark call _forward_backbone directly, bypassing Module.__call__.
            # Last attention receives the same cache and executes after all previous layers.
            if hasattr(draft, "_forward_backbone"):
                modules.append(("draft", draft.layers[-1].self_attn))
        for role, module in modules:
            handles.append(module.register_forward_pre_hook(make_pre(role), with_kwargs=True))
            handles.append(module.register_forward_hook(make_post(role), with_kwargs=True))
        original_crop = DynamicCache.crop
        recorder = self
        def crop(cache, *args, **kwargs):
            result = original_crop(cache, *args, **kwargs)
            if any(ref() is cache for ref in recorder.refs.values()):
                recorder.sample("after_crop")
            return result
        DynamicCache.crop = crop
        try:
            yield
        finally:
            DynamicCache.crop = original_crop
            for handle in handles:
                handle.remove()


@torch.inference_mode()
def profile_generation(method, target, draft, input_ids, max_new_tokens=64):
    """Run the ready upstream generation loop; measure live cache checkpoints."""
    recorder = CacheRecorder()
    with recorder.instrument(target, draft):
        if method in ("dflash", "dspark", "eagle3"):
            evaluator = make_evaluator(method, target, draft, max_new_tokens)
            result = evaluator.generate_one_sample(input_ids=input_ids, stop_token_ids=None)
            output = result.output_ids
        else:
            options = dict(max_new_tokens=max_new_tokens, do_sample=False,
                           eos_token_id=None, pad_token_id=0)
            if method == "orthrus":
                # Official implementation expects a scalar EOS, not a list.
                output = target.generate(input_ids=input_ids,
                            max_new_tokens=max_new_tokens, temperature=0.0,
                            eos_token_id=-1, use_diffusion_mode=True)
            elif method == "ar_draft":
                output = target.generate(input_ids=input_ids, assistant_model=draft, **options)
            else:
                output = target.generate(input_ids=input_ids, **options)
    if not recorder.trace:
        raise AssertionError("No real KV-cache samples were recorded")
    last = recorder.trace[-1]
    row = {k: v for k, v in last.items() if k not in ("event", "sample")}
    row.update(mode="generation", method=method, context_tokens=int(input_ids.shape[1]),
               generated_tokens=int(output.shape[1] - input_ids.shape[1]), status="ok")
    for key in ("target_tokens", "draft_tokens", "target_bytes", "draft_bytes",
                "total_bytes", "total_storage_bytes"):
        row["peak_" + key] = max(sample[key] for sample in recorder.trace)
    row["output_tokens"] = output[0].detach().cpu().tolist()
    return row, recorder.trace


def load_models(settings, method, device):
    specs = settings["models"]
    target_spec = specs["orthrus"] if method == "orthrus" else specs["target"]
    dtype = getattr(torch, settings["dtype"])
    common = dict(dtype=dtype, attn_implementation="sdpa", low_cpu_mem_usage=True)
    target = AutoModelForCausalLM.from_pretrained(target_spec["id"],
                revision=target_spec.get("revision"), trust_remote_code=method == "orthrus",
                **common).to(device).eval()
    target.config.use_cache = True
    target.generation_config.use_cache = True
    tokenizer = AutoTokenizer.from_pretrained(specs["target"]["id"],
                revision=specs["target"].get("revision"))
    draft = None
    if method in ("dflash", "dspark", "eagle3"):
        sys.path.insert(0, settings["deepspec_dir"])
        if method == "eagle3":
            from deepspec.modeling.eagle3.qwen3 import Qwen3Eagle3Model
            cls = Qwen3Eagle3Model
        else:
            from deepspec.modeling.dspark.qwen3 import Qwen3DSparkModel
            cls = Qwen3DSparkModel
        draft_spec = specs[method]
        draft = cls.from_pretrained(draft_spec["id"],
                    revision=draft_spec.get("revision"), **common).to(device).eval()
    elif method == "ar_draft":
        spec = specs["ar_draft"]
        draft = AutoModelForCausalLM.from_pretrained(spec["id"],
                    revision=spec.get("revision"), **common).to(device).eval()
        other_tokenizer = AutoTokenizer.from_pretrained(spec["id"], revision=spec.get("revision"))
        if other_tokenizer.get_vocab() != tokenizer.get_vocab():
            raise ValueError("AR drafter vocabulary differs from target; explicit UAD is required")
        draft.generation_config.num_assistant_tokens = settings.get("ar_draft_tokens", 7)
        draft.generation_config.num_assistant_tokens_schedule = "constant"
    return target, draft, tokenizer


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--settings", required=True)
    parser.add_argument("--method", required=True)
    parser.add_argument("--mode", choices=["fixed_prefix", "generation"], default="fixed_prefix")
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    settings = json.loads(Path(args.settings).read_text())
    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available in the worker environment")
    torch.set_num_threads(4)
    torch.manual_seed(42)
    device = torch.device("cuda:0")
    target, draft, tokenizer = load_models(settings, args.method, device)
    token_seed = tokenizer.encode(settings["prefix_text"], add_special_tokens=False)
    if not token_seed:
        raise ValueError("prefix_text must tokenize to a nonempty sequence")
    rows = []
    for n in settings["contexts"]:
        # Identical token ids and exact length for every method; a memory probe, not a quality test.
        ids = (token_seed * ((n + len(token_seed) - 1) // len(token_seed)))[:n]
        input_ids = torch.tensor([ids], device=device)
        print(f"[{args.method}] {args.mode}: {n} context tokens", flush=True)
        gc.collect()
        torch.cuda.empty_cache()
        if args.mode == "fixed_prefix":
            row = measure_prefix(args.method, target, draft, input_ids)
        else:
            row, trace = profile_generation(args.method, target, draft, input_ids,
                                settings["max_new_tokens"])
            trace_path = out_path.with_name(out_path.stem + f"-T{n}-trace.json")
            trace_path.write_text(json.dumps(trace, indent=2))
        row.update(target_checkpoint=settings["models"]["orthrus" if args.method == "orthrus" else "target"]["id"],
                   draft_checkpoint=settings["models"].get(args.method, {}).get("id")
                   if draft is not None else None,
                   dtype=settings["dtype"], gpu=torch.cuda.get_device_name(device),
                   transformers=importlib.import_module("transformers").__version__,
                   torch_version=torch.__version__, measured_at=time.strftime("%Y-%m-%dT%H:%M:%S"))
        rows.append(row)
        out_path.write_text(json.dumps(rows, indent=2))
        print(json.dumps({k: row[k] for k in ("method", "context_tokens", "target_tokens", "draft_tokens", "total_bytes")}), flush=True)
    del target, draft
    gc.collect()
    torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

'''
WORKER_PATH = ROOT / 'kv_worker.py'
WORKER_PATH.write_text(WORKER_SOURCE.lstrip(), encoding='utf-8')
print('Измеритель:', WORKER_PATH)

## 4. Расчётный ориентир — не результаты GPU-замера
Для обычного полного KV-кэша: `2 * layers * kv_heads * head_dim * dtype_bytes * N`.
Снимки конфигов ниже соответствуют закреплённым ревизиям default checkpoints.
Если меняешь checkpoint, ориентир для старого снимка будет пропущен.

In [ ]:
CONFIG_SNAPSHOTS = {'target': {'architectures': ['Qwen3ForCausalLM'], 'attention_bias': False, 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 12288, 'max_position_embeddings': 40960, 'max_window_layers': 36, 'model_type': 'qwen3', 'num_attention_heads': 32, 'num_hidden_layers': 36, 'num_key_value_heads': 8, 'rms_norm_eps': 1e-06, 'rope_scaling': None, 'rope_theta': 1000000, 'sliding_window': None, 'tie_word_embeddings': False, 'torch_dtype': 'bfloat16', 'transformers_version': '4.51.0', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936}, 'orthrus': {'architectures': ['OrthrusLM'], 'attention_bias': False, 'attention_dropout': 0.0, 'auto_map': {'AutoModelForCausalLM': 'modeling_orthrus.OrthrusLM'}, 'block_size': 32, 'bos_token_id': 151643, 'dtype': 'bfloat16', 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 12288, 'layer_types': ['full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention'], 'lora_alpha': 32, 'mask_token_id': 151669, 'max_position_embeddings': 40960, 'max_window_layers': 36, 'model_type': 'qwen3', 'num_attention_heads': 32, 'num_hidden_layers': 36, 'num_key_value_heads': 8, 'pad_token_id': None, 'r': 16, 'rms_norm_eps': 1e-06, 'rope_parameters': {'rope_theta': 1000000, 'rope_type': 'default'}, 'sliding_window': None, 'tie_word_embeddings': False, 'transformers_version': '5.8.1', 'use_cache': False, 'use_sliding_window': False, 'vocab_size': 151936}, 'dflash': {'architectures': ['Qwen3DSparkModel'], 'attention_bias': False, 'attention_dropout': 0.0, 'block_size': 7, 'bos_token_id': 151643, 'dtype': 'bfloat16', 'enable_confidence_head': False, 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 12288, 'layer_types': ['full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention'], 'markov_rank': 0, 'mask_token_id': 151669, 'max_position_embeddings': 40960, 'max_window_layers': 36, 'model_type': 'qwen3', 'num_anchors': 512, 'num_attention_heads': 32, 'num_hidden_layers': 5, 'num_key_value_heads': 8, 'num_target_layers': 36, 'pad_token_id': None, 'rms_norm_eps': 1e-06, 'rope_parameters': {'rope_theta': 1000000, 'rope_type': 'default'}, 'sliding_window': None, 'target_layer_ids': [1, 9, 17, 25, 33], 'tie_word_embeddings': False, 'transformers_version': '5.10.2', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936}, 'dspark': {'architectures': ['Qwen3DSparkModel'], 'attention_bias': False, 'attention_dropout': 0.0, 'block_size': 7, 'bos_token_id': 151643, 'confidence_head_with_markov': True, 'dtype': 'bfloat16', 'enable_confidence_head': True, 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 12288, 'layer_types': ['full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention'], 'markov_head_type': 'vanilla', 'markov_rank': 256, 'mask_token_id': 151669, 'max_position_embeddings': 40960, 'max_window_layers': 36, 'model_type': 'qwen3', 'num_anchors': 512, 'num_attention_heads': 32, 'num_hidden_layers': 5, 'num_key_value_heads': 8, 'num_target_layers': 36, 'pad_token_id': None, 'rms_norm_eps': 1e-06, 'rope_parameters': {'rope_theta': 1000000, 'rope_type': 'default'}, 'sliding_window': None, 'target_layer_ids': [1, 9, 17, 25, 33], 'tie_word_embeddings': False, 'transformers_version': '5.10.2', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936}, 'eagle3': {'architectures': ['Qwen3Eagle3Model'], 'attention_bias': False, 'attention_dropout': 0.0, 'bos_token_id': 151643, 'draft_num_hidden_layers': 1, 'dtype': 'bfloat16', 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 12288, 'layer_types': ['full_attention'], 'max_position_embeddings': 40960, 'max_window_layers': 36, 'model_type': 'qwen3', 'num_attention_heads': 32, 'num_hidden_layers': 1, 'num_key_value_heads': 8, 'num_target_layers': 36, 'pad_token_id': None, 'rms_norm_eps': 1e-06, 'rope_parameters': {'rope_theta': 1000000, 'rope_type': 'default'}, 'sliding_window': None, 'step_loss_decay': 0.8, 'target_layer_ids': [1, 9, 17, 25, 33], 'target_model_name_or_path': 'Qwen/Qwen3-8B', 'tie_word_embeddings': False, 'transformers_version': '5.10.2', 'ttt_length': 7, 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936}, 'ar_draft': {'architectures': ['Qwen3ForCausalLM'], 'attention_bias': False, 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'head_dim': 128, 'hidden_act': 'silu', 'hidden_size': 1024, 'initializer_range': 0.02, 'intermediate_size': 3072, 'max_position_embeddings': 40960, 'max_window_layers': 28, 'model_type': 'qwen3', 'num_attention_heads': 16, 'num_hidden_layers': 28, 'num_key_value_heads': 8, 'rms_norm_eps': 1e-06, 'rope_scaling': None, 'rope_theta': 1000000, 'sliding_window': None, 'tie_word_embeddings': True, 'torch_dtype': 'bfloat16', 'transformers_version': '4.51.0', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936}}
SNAPSHOT_MODELS = {'target': {'id': 'Qwen/Qwen3-8B', 'revision': 'b968826d9c46dd6066d109eabc6255188de91218'}, 'orthrus': {'id': 'chiennv/Orthrus-Qwen3-8B', 'revision': '977a617772e91c966a8cd9b551f4151f9824b6fa'}, 'dflash': {'id': 'deepseek-ai/dflash_qwen3_8b_block7', 'revision': '9e44dbbb6cb68b0c943abf9c5fc3c17c00897cdf'}, 'dspark': {'id': 'deepseek-ai/dspark_qwen3_8b_block7', 'revision': '03326e5043815da1f81b109078b2889737c26017'}, 'eagle3': {'id': 'deepseek-ai/eagle3_qwen3_8b_ttt7', 'revision': 'f6485ba8d21e11942958617dbe7e71b467f38f38'}, 'ar_draft': {'id': 'Qwen/Qwen3-0.6B', 'revision': 'c1899de289a04d12100db370d81485cdf75e47ca'}}
import pandas as pd
from IPython.display import display

LABELS = {
    "ar": "Qwen3-8B / AR", "orthrus": "Qwen3-8B / Orthrus",
    "dflash": "Qwen3-8B / DFlash", "dspark": "Qwen3-8B / DSpark",
    "eagle3": "Qwen3-8B / EAGLE-3", "ar_draft": "Qwen3-8B + AR Qwen3-0.6B",
}
def config_bytes_per_token(config, dtype):
    item_size = {"bfloat16": 2, "float16": 2, "float32": 4}[dtype]
    head_dim = config.get("head_dim") or config["hidden_size"] // config["num_attention_heads"]
    return 2 * config["num_hidden_layers"] * config["num_key_value_heads"] * head_dim * item_size

expected_rows = []
for method in METHODS:
    target_key = "orthrus" if method == "orthrus" else "target"
    draft_key = method if method in ["dflash", "dspark", "eagle3", "ar_draft"] else None
    keys = [target_key] + ([draft_key] if draft_key else [])
    if any(MODELS[k] != SNAPSHOT_MODELS[k] for k in keys):
        print("Нет снимка нового конфига:", method)
        continue
    for n in CONTEXTS:
        target_bytes = n * config_bytes_per_token(CONFIG_SNAPSHOTS[target_key], DTYPE)
        draft_bytes = n * config_bytes_per_token(CONFIG_SNAPSHOTS[draft_key], DTYPE) if draft_key else 0
        expected_rows.append({
            "method": method, "model": LABELS[method], "context_tokens": n,
            "target_tokens": n, "draft_tokens": n if draft_key else 0,
            "target_bytes": target_bytes, "draft_bytes": draft_bytes,
            "total_bytes": target_bytes + draft_bytes,
            "total_MiB": (target_bytes + draft_bytes) / 1024**2,
            "extra_KV_percent": 100 * draft_bytes / target_bytes,
            "source": "calculated_from_pinned_configs",
        })
expected_df = pd.DataFrame(expected_rows, columns=[
    "method", "model", "context_tokens", "target_tokens", "draft_tokens",
    "target_bytes", "draft_bytes", "total_bytes", "total_MiB", "extra_KV_percent", "source",
])
display(expected_df[expected_df.context_tokens == 2048] if 2048 in CONTEXTS else expected_df)

## 5. Реальные замеры всех шести вариантов
Порядок: target prefill → инициализация draft/history через upstream-вызовы → снимок кэшей
строго на N позициях. DFlash/DSpark строят кэш при первом proposal и обрезают mask-блок;
EAGLE-3 инициализирует кэш из target features, затем его пробный proposal обрезается до N.
Orthrus выполняет diffusion-проход с общим cache: отдельная история не создаётся.

Каждый метод запускается в отдельном процессе. Веса скачиваются через общий HF cache и
используются повторно. Уже завершённые строки в этой конфигурации можно переиспользовать.
Ошибки/OOM показываются как `status=error`, а не как нулевой кэш.

In [ ]:
settings = {
    "models": MODELS, "contexts": CONTEXTS, "dtype": DTYPE,
    "worker_sha256": hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    "deepspec_dir": str(DEEPSPEC_DIR), "deepspec_revision": DEEPSPEC_REVISION,
    "max_new_tokens": MAX_NEW_TOKENS, "ar_draft_tokens": 7,
    "prefix_text": """
    Solve the following problem step by step.

    A box contains 5 red balls, 4 blue balls, and 3 green balls.
    Four balls are drawn uniformly at random without replacement.

    What is the probability that exactly two of the drawn balls
    are red and at least one is blue?

    Give your answer as a simplified fraction.""",
}
experiment_hash = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
RESULT_DIR = ROOT / "results" / experiment_hash
RESULT_DIR.mkdir(parents=True, exist_ok=True)
SETTINGS_PATH = RESULT_DIR / "settings.json"
SETTINGS_PATH.write_text(json.dumps(settings, indent=2), encoding="utf-8")


In [ ]:

def run_methods(mode="fixed_prefix", rerun=False):
    for method in METHODS:
        result_path = RESULT_DIR / f"{mode}-{method}.json"
        if not rerun and result_path.exists():
            old_rows = json.loads(result_path.read_text())
            if {r.get("context_tokens") for r in old_rows} == set(CONTEXTS):
                print("Готовые измерения:", method, mode)
                continue
        command = [str(PYTHON), str(WORKER_PATH), "--settings", str(SETTINGS_PATH),
                   "--method", method, "--mode", mode, "--out", str(result_path)]
        log_path = RESULT_DIR / f"{mode}-{method}.log"
        print("Запуск:", LABELS[method], flush=True)
        with log_path.open("w", encoding="utf-8") as log:
            proc = subprocess.Popen(command, env=worker_env, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                log.write(line)
                log.flush()
                if line.startswith("[") or line.startswith('{"method"'):
                    print(line.rstrip(), flush=True)
            rc = proc.wait()
        error_path = RESULT_DIR / f"{mode}-{method}-error.json"
        if rc:
            tail = log_path.read_text(errors="replace")[-3000:]
            error_path.write_text(json.dumps({"returncode": rc, "log": str(log_path), "tail": tail}))
            print("Ошибка:", method, "—", log_path)
            print(tail)
        else:
            error_path.unlink(missing_ok=True)
    print("Результаты:", RESULT_DIR)

run_methods("fixed_prefix")

## 6. Сводная таблица реальных результатов
Все методы и длины присутствуют в таблице. Отсутствующее измерение не подменяется расчётом.
CSV содержит точные байты, число позиций и сведения о checkpoint/GPU/версиях.

In [ ]:
def collect_results(mode):
    rows = []
    for method in METHODS:
        path = RESULT_DIR / f"{mode}-{method}.json"
        data = json.loads(path.read_text()) if path.exists() else []
        by_n = {r["context_tokens"]: r for r in data}
        error = RESULT_DIR / f"{mode}-{method}-error.json"
        for n in CONTEXTS:
            row = by_n.get(n, {"method": method, "context_tokens": n,
                              "status": "error" if error.exists() else "not_run"})
            rows.append({**row, "model": LABELS[method]})
    df = pd.DataFrame(rows)
    # Columns remain available even when every worker failed.
    columns = ["target_tokens", "draft_tokens", "sum_cache_positions", "target_layer_slots",
               "draft_layer_slots", "target_bytes", "draft_bytes", "total_bytes",
               "target_storage_bytes", "draft_storage_bytes", "total_storage_bytes"]
    for name in columns:
        if name not in df:
            df[name] = pd.NA
    for name in ["target", "draft", "total"]:
        df[name + "_MiB"] = pd.to_numeric(df[name + "_bytes"], errors="coerce") / 1024**2
    df["total_storage_MiB"] = pd.to_numeric(df.total_storage_bytes, errors="coerce") / 1024**2
    df["extra_KV_percent"] = 100 * pd.to_numeric(df.draft_bytes, errors="coerce") / pd.to_numeric(df.target_bytes, errors="coerce")
    return df

summary_df = collect_results("fixed_prefix")
table_columns = ["model", "context_tokens", "status", "target_tokens", "draft_tokens",
                 "sum_cache_positions", "target_bytes", "draft_bytes", "total_bytes",
                 "target_MiB", "draft_MiB", "total_MiB", "total_storage_MiB", "extra_KV_percent"]
display(summary_df[table_columns])
summary_df.to_csv(RESULT_DIR / "kv_summary_measured.csv", index=False)
expected_df.to_csv(RESULT_DIR / "kv_summary_calculated.csv", index=False)

# Проверка совпадения измеренных payload bytes с ориентиром, когда checkpoint не менялся.
if not expected_df.empty:
    checks = summary_df.merge(expected_df[["method", "context_tokens", "total_bytes"]],
                              on=["method", "context_tokens"], suffixes=("_measured", "_expected"))
    checks = checks[checks.status == "ok"]
    if not checks.empty:
        checks["matches_formula"] = checks.total_bytes_measured == checks.total_bytes_expected
        display(checks[["model", "context_tokens", "matches_formula"]])
        assert checks.matches_formula.all(), "Измеренные байты отличаются от конфигов — проверь dtype/shape"

## 7. Графики по измеренным значениям
Если какой-то вариант не запустился, его столбец не рисуется: таблица выше сохраняет ошибку.

In [ ]:
import matplotlib.pyplot as plt

ok = summary_df[summary_df.status == "ok"].copy()
if not ok.empty:
    selected_n = 2048 if 2048 in CONTEXTS else CONTEXTS[0]
    at_n = ok[ok.context_tokens == selected_n]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    x = range(len(at_n))
    axes[0].bar(x, at_n.target_MiB, label="Target KV", color="#4F76C7")
    axes[0].bar(x, at_n.draft_MiB, bottom=at_n.target_MiB, label="Draft KV", color="#E4A044")
    axes[0].set_xticks(list(x), at_n.method, rotation=25)
    axes[0].set_ylabel("KV payload, MiB")
    axes[0].set_title(f"Measured persistent KV: {selected_n} prefix tokens")
    axes[0].legend()
    for method, frame in ok.groupby("method", sort=False):
        frame = frame.sort_values("context_tokens")
        axes[1].plot(frame.context_tokens, frame.total_MiB, marker="o", label=method)
    axes[1].set_xlabel("Prefix tokens")
    axes[1].set_ylabel("Target + draft KV, MiB")
    axes[1].set_title("Measured persistent KV vs context length")
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / "kv_measured.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
else:
    print("Нет GPU-измерений для графика. Посмотри status и логи в RESULT_DIR.")

## 8. Дополнительно: готовая генерация и пик объектов кэша
Поставь `RUN_GENERATION=True` в ячейке 1 и выполни эту ячейку. Используются
upstream `generate`, Transformers `assistant_model` и DeepSpec `generate_one_sample`.
Это **другой эксперимент**: число принятых токенов/размер последнего блока могут отличаться.
Не смешивай его финальные длины с фиксированным префиксом в основной таблице.
`peak_total_bytes` — максимум одновременно живых измеренных K/V в точках после forward/crop,
а не сумма независимых пиков target/draft и не общий CUDA peak.
`eos` отключён для фиксированного лимита токенов; это memory probe, не чат-оценка.

In [ ]:
MAX_NEW_TOKENS = 512
RUN_GENERATION = True

settings["max_new_tokens"] = MAX_NEW_TOKENS

# Новая папка, чтобы не переиспользовать результаты с лимитом 64.
experiment_hash = hashlib.sha256(
    json.dumps(settings, sort_keys=True).encode()
).hexdigest()[:12]

RESULT_DIR = ROOT / "results" / experiment_hash
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_PATH = RESULT_DIR / "settings.json"
SETTINGS_PATH.write_text(
    json.dumps(settings, indent=2),
    encoding="utf-8",
)

In [ ]:
if RUN_GENERATION:
    run_methods("generation")
    generation_df = collect_results("generation")
    for col in ["generated_tokens", "peak_target_tokens", "peak_draft_tokens", "peak_total_bytes"]:
        if col not in generation_df:
            generation_df[col] = pd.NA
    generation_df["peak_total_MiB"] = pd.to_numeric(generation_df.peak_total_bytes, errors="coerce") / 1024**2
    display(generation_df[["model", "context_tokens", "status", "generated_tokens",
                           "target_tokens", "draft_tokens", "peak_target_tokens",
                           "peak_draft_tokens", "target_bytes", "draft_bytes",
                           "total_bytes", "peak_total_bytes", "peak_total_MiB"]])
    generation_df.to_csv(RESULT_DIR / "kv_generation_measured.csv", index=False)
    # Сверка выводов: различия в bf16 возможны из-за округления AR logits.
    valid_gen = generation_df[generation_df.status == "ok"]
    for n, frame in valid_gen.groupby("context_tokens"):
        reference = frame[frame.method == "ar"]
        if len(reference):
            ref = reference.iloc[0].output_tokens
            print("Контекст", n, "— совпадение с AR:",
                  {r.method: r.output_tokens == ref for r in frame.itertuples()})
else:
    print("Основная таблица уже готова. Для полного decoder probe включи RUN_GENERATION.")

In [ ]:
import pandas as pd
from IPython.display import display

table = generation_df.copy()

for name in ["target", "draft", "total"]:
    table[f"{name}_MiB"] = (
        pd.to_numeric(table[f"{name}_bytes"], errors="coerce") / 1024**2
    )

table["peak_total_MiB"] = (
    pd.to_numeric(table["peak_total_bytes"], errors="coerce") / 1024**2
)

columns = [
    "model", "context_tokens", "status", "generated_tokens",
    "target_tokens", "draft_tokens",
    "target_MiB", "draft_MiB", "total_MiB", "peak_total_MiB",
]

display(
    table[columns].style
    .format({
        "target_MiB": "{:.2f}",
        "draft_MiB": "{:.2f}",
        "total_MiB": "{:.2f}",
        "peak_total_MiB": "{:.2f}",
    }, na_rep="—")
    .hide(axis="index")
)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

N = 2048

rows = []

for method in METHODS[:-1]:
    path = RESULT_DIR / f"generation-{method}-T{N}-trace.json"

    if not path.exists():
        print(f"Нет истории для {method}")
        continue

    trace = json.loads(path.read_text())
    if not trace:
        continue

    peak = max(trace, key=lambda point: point["total_bytes"])

    rows.append({
        "method": method,
        "target_MiB": peak["target_bytes"] / 1024**2,
        "draft_MiB": peak["draft_bytes"] / 1024**2,
    })

peak_df = pd.DataFrame(rows)

if peak_df.empty:
    print("Сначала запусти раздел 8 с RUN_GENERATION=True.")
else:
    x = np.arange(len(peak_df))
    total = peak_df.target_MiB + peak_df.draft_MiB

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.bar(
        x, peak_df.target_MiB,
        label="Target KV", color="#4F76C7",
    )
    ax.bar(
        x, peak_df.draft_MiB,
        bottom=peak_df.target_MiB,
        label="Draft KV", color="#E4A044",
    )

    for i, value in enumerate(total):
        ax.text(
            i, value + total.max() * 0.015,
            f"{value:.1f}", ha="center", fontsize=10,
        )

    ax.set_xticks(x, peak_df.method, rotation=20)
    ax.set_ylabel("Объём KV-кэша, MiB")
    ax.set_title(
        f"Пиковый KV-кэш при генерации\n"
        f"Префикс: {N} токенов, лимит ответа: {MAX_NEW_TOKENS}"
    )
    ax.set_ylim(0, total.max() * 1.15)
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    fig.tight_layout()
    fig.savefig(
        RESULT_DIR / f"kv_generation_peak_T{N}.png",
        dpi=200, bbox_inches="tight",
    )
    plt.show()

In [ ]:
import json
import subprocess
import pandas as pd
from IPython.display import display

SPEED_CONTEXTS = [2048]    # Можно поставить CONTEXTS
SPEED_NEW_TOKENS = 512
SPEED_REPEATS = 3

# Использует WORKER_PATH, PYTHON, worker_env и settings из ноутбука.
speed_worker = r'''
import importlib.util
import json
import statistics
import sys
import time

import torch

payload = json.load(sys.stdin)
spec = importlib.util.spec_from_file_location(
    "kv_speed_worker", payload["worker_path"]
)
w = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = w
spec.loader.exec_module(w)

settings = payload["settings"]
method = payload["method"]
limit = payload["new_tokens"]

torch.set_num_threads(4)
torch.manual_seed(42)
device = torch.device("cuda:0")
target, draft, tokenizer = w.load_models(settings, method, device)

# Одинаковое число новых токенов: не завершаемся по EOS.
target.generation_config.eos_token_id = None
if method == "ar_draft":
    draft.generation_config.eos_token_id = None

evaluator = (
    w.make_evaluator(method, target, draft, limit)
    if method in ("dflash", "dspark", "eagle3")
    else None
)

@torch.inference_mode()
def generate(ids):
    if evaluator is not None:
        return evaluator.generate_one_sample(
            input_ids=ids, stop_token_ids=None
        ).output_ids

    if method == "orthrus":
        return target.generate(
            input_ids=ids,
            max_new_tokens=limit,
            temperature=0.0,
            eos_token_id=-1,
            use_diffusion_mode=True,
        )

    kwargs = dict(
        input_ids=ids,
        max_new_tokens=limit,
        do_sample=False,
        eos_token_id=None,
        pad_token_id=0,
    )
    if method == "ar_draft":
        kwargs["assistant_model"] = draft
    return target.generate(**kwargs)

seed = tokenizer.encode(settings["prefix_text"], add_special_tokens=False)
rows = []

for n in payload["contexts"]:
    tokens = (seed * ((n + len(seed) - 1) // len(seed)))[:n]
    ids = torch.tensor([tokens], device=device)

    # Прогрев; затем отдельный проход со счётчиком для TPF.
    output = generate(ids)
    del output
    torch.cuda.synchronize()

    calls = [0]
    def count_forward(module, args):
        calls[0] += 1

    handle = target.register_forward_pre_hook(count_forward)
    try:
        output = generate(ids)
        generated = output.shape[1] - n
        del output
        torch.cuda.synchronize()
    finally:
        handle.remove()

    # Замеры времени без hooks и измерителей кэша.
    times = []
    for _ in range(payload["repeats"]):
        torch.cuda.synchronize()
        started = time.perf_counter()
        output = generate(ids)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started

        assert output.shape[1] - n == generated
        times.append(elapsed)
        del output

    seconds = statistics.median(times)
    rows.append({
        "method": method,
        "context_tokens": n,
        "generated_tokens": generated,
        "target_forwards": calls[0],
        "TPF": generated / calls[0],
        "seconds": seconds,
        "tokens/sec": generated / seconds,
        "status": "ok",
    })

print("SPEED_JSON=" + json.dumps(rows), flush=True)
'''

speed_rows = []

for method in METHODS:
    print(f"Замер: {LABELS[method]}", flush=True)
    payload = {
        "worker_path": str(WORKER_PATH.resolve()),
        "settings": settings,
        "method": method,
        "contexts": SPEED_CONTEXTS,
        "new_tokens": SPEED_NEW_TOKENS,
        "repeats": SPEED_REPEATS,
    }
    proc = subprocess.run(
        [str(PYTHON), "-c", speed_worker],
        input=json.dumps(payload),
        env=worker_env,
        text=True,
        capture_output=True,
    )
    result = next(
        (line.removeprefix("SPEED_JSON=")
         for line in reversed(proc.stdout.splitlines())
         if line.startswith("SPEED_JSON=")),
        None,
    )
    if proc.returncode == 0 and result is not None:
        speed_rows.extend(json.loads(result))
    else:
        print((proc.stderr or proc.stdout)[-2000:])
        speed_rows.extend({
            "method": method, "context_tokens": n, "status": "error"
        } for n in SPEED_CONTEXTS)

speed_df = pd.DataFrame(speed_rows)
for column in ["generated_tokens", "target_forwards", "TPF",
               "seconds", "tokens/sec"]:
    if column not in speed_df:
        speed_df[column] = float("nan")

ar_seconds = speed_df[
    (speed_df.method == "ar") & (speed_df.status == "ok")
].set_index("context_tokens")["seconds"]

speed_df["speedup_vs_AR"] = (
    speed_df.context_tokens.map(ar_seconds) / speed_df.seconds
)
speed_df["model"] = speed_df.method.map(LABELS)

display(
    speed_df[[
        "model", "context_tokens", "status", "generated_tokens",
        "target_forwards", "TPF", "seconds", "tokens/sec", "speedup_vs_AR",
    ]].style.format({
        "TPF": "{:.2f}",
        "seconds": "{:.3f}",
        "tokens/sec": "{:.2f}",
        "speedup_vs_AR": "{:.2f}×",
    }, na_rep="—").hide(axis="index")
)

## Интерпретация и границы вывода
При одном и том же N у default checkpoints ожидается одинаковый target KV (36 слоёв).
У Orthrus нет второй постоянной истории. У DeepSpec DFlash/DSpark есть draft KV на 5 слоях,
у EAGLE-3 — на 1 слое, у AR Qwen3-0.6B — на 28 слоях.
Это сравнение выбранных опубликованных реализаций; оно не доказывает такой же расход
у всех возможных tree/window/quantized/shared-KV реализаций.

KV bytes не включают веса, target features, logits и transient attention concatenations.
В частности, Orthrus временно вычисляет K/V блока и может создавать `torch.cat` буферы.
Отсутствие второго постоянного cache не означает нулевой overhead общей GPU-памяти.
Модель из worker освобождается при завершении процесса; уже загруженная в notebook модель
остаётся в kernel. CSV/JSON/PNG и скачанные веса остаются на диске.

**Источники:**
- [DeepSpec: код и выпущенные Qwen3-8B drafts](https://github.com/deepseek-ai/DeepSpec/tree/005e03b81cec38b7da6399833d609ee89a2587f2)
- [Orthrus-Qwen3-8B](https://huggingface.co/chiennv/Orthrus-Qwen3-8B)
- [Transformers assisted decoding](https://huggingface.co/docs/transformers/generation_strategies#assisted-decoding)
- [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B)

**Проверка при подготовке:** Torch 2.9.1 CPU + Transformers 5.10.2; все шесть вариантов
прошли real forward/prefill, проверку payload bytes, crop/view и storage alias accounting.
Готовые decoder loops всех шести вариантов совпали с AR на маленьких random-init моделях
в fp32. Реальные 8B checkpoints на GPU в этой среде не запускались.